## Setup

Prepare your environment the same way as in the module's
[Environment](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/02-environment.md) lesson.

This homework needs one extra library: `gitsource`, which downloads files
from a GitHub repository.

Install it:

```bash
uv add gitsource
```

For the LLM, we recommend OpenAI with `gpt-5.4-mini`, but you can use any model
and provider you like - just adapt the client and the usage fields accordingly.

## Preparation

First, we will pull the lesson pages straight from the course repository. 
We will use the commit `8c1834d` to make sure everyone works with the exact same data.

We will use `gitsource` for that:

```python
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
```

`GithubRepositoryDataReader` downloads the entire repository and goes over all the files in it. Because we specify `allowed_extensions={"md"}`, it only checks  the markdown files.

We also pass a `filename_filter` so we don't grab every markdown file in the
repo, like the top-level README. The lesson pages all live under a module's `lessons/` folder, so
filtering on `/lessons/` keeps just those.


Each file has a `parse()` method that returns a dictionary with its
`filename` and `content`:

```python
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
```

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [7]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

How many lesson pages are in the dataset?

In [10]:
len(documents)

72

## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

In [11]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question= "How does the agentic loop keep calling the model until it stops?"

index.search(query=question, num_results=1)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

In [13]:
from rag_helper import RAGBase

class CustomRAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)
    
    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(f"Content: {doc['content']}")
            lines.append("")

        return '\n'.join(lines).strip()

In [22]:
import tiktoken
from openai import OpenAI

llm_client = OpenAI()

rag_assistant = CustomRAG(
    index=index,
    llm_client=llm_client,
    model='gpt-5.4-mini'
)

query = "How does the agentic loop keep calling the model until it stops?"

search_results = rag_assistant.search(query, num_results=1)

prompt = rag_assistant.build_prompt(query, search_results)

encoding = tiktoken.get_encoding("cl100k_base")
prompt_tokens = len(encoding.encode(prompt))

prompt_tokens

2318

In [23]:
answer = rag_assistant.llm(prompt)
answer

'The loop keeps calling the model by checking whether the latest response contains any function calls.\n\nHow it works:\n1. Send the current `messages` to the model.\n2. If the model returns a `function_call`, run the tool and append the tool result to `messages`.\n3. Set `has_function_calls = True`.\n4. Repeat the `while True` loop.\n5. Stop only when a response comes back with **no function calls**.\n\nSo the exit condition is:\n\n- **No function calls this turn → break out of the loop**\n\nThat’s how it keeps going until the model stops asking for tools.'

## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

In [24]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [27]:
len(chunks)

295

## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

In [28]:
index_chunked = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunked.fit(chunks)

In [29]:
llm_client = OpenAI()

rag_assistant = CustomRAG(
    index=index_chunked,
    llm_client=llm_client,
    model='gpt-5.4-mini'
)

query = "How does the agentic loop keep calling the model until it stops?"

search_results = rag_assistant.search(query, num_results=1)

prompt = rag_assistant.build_prompt(query, search_results)

encoding = tiktoken.get_encoding("cl100k_base")
prompt_tokens = len(encoding.encode(prompt))

prompt_tokens

494

## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

```bash
uv add toyaikit
```

Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

In [30]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()

In [31]:
def search(query):
    results = rag_assistant.search(query, num_results=3) 
    context = rag_assistant.build_context(results)
    return context

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the lesson pages database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the lesson pages."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

agent_tools.add_tool(search, search_tool)

In [32]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [33]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [35]:
result.cost, result.all_messages

(CostInfo(input_cost=Decimal('0.00371775'), output_cost=Decimal('0.0020835'), total_cost=Decimal('0.00580125')),
 [EasyInputMessage(content="\nYou're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None),
  EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
  ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG difference"}', call_id='call_CswpPbtMD2vKJXoeNopMVQyH', name='search', type='function_call', id='fc_01c1ee33da335c33006a2dd6e856e4819b819eef8cf0615366', namespace=None, status='completed'),
  ResponseFunctionToolCall(arguments='{"query":"agentic loop iterative retrieval reasoning"}', call_id='call_szn4iju3bkWItzM04UHblijo', name='search', type='function_call', id='fc_01c1ee33da335c33006a2dd6e85704819bb200cecc8df5eadc', namespace=None, status='comple